# RunPod GPU 환경용 KoBERT 기반 2차 모델 학습
## 맥락 기반 보이스피싱 탐지 모델 - GPU 최적화 버전

## 1. 환경 설정 및 라이브러리 설치

In [ ]:
# RunPod 환경에 필요한 라이브러리 설치
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets tokenizers
!pip install scikit-learn pandas numpy matplotlib seaborn
!pip install tqdm ipywidgets

# GPU 메모리 최적화를 위한 추가 설정
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# GPU 설정 및 메모리 최적화
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

if torch.cuda.is_available():
    print(f'GPU Name: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    
    # GPU 메모리 정리
    torch.cuda.empty_cache()
    
    # Mixed precision 설정
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    use_amp = True
else:
    use_amp = False
    print('CUDA is not available. Using CPU.')

## 2. 데이터 업로드 및 로딩
**주의**: RunPod 환경에서는 다음 파일들을 업로드해야 합니다:
- `master_dataset_final.csv` (훈련용 데이터)
- `1차모델_테스트데이터셋.csv` (테스트용 데이터)

In [ ]:
# 파일 업로드 확인
import os

# 현재 작업 디렉토리 확인
print(f"현재 작업 디렉토리: {os.getcwd()}")
print("\n현재 디렉토리의 파일들:")
for file in os.listdir('.'):
    print(f"  - {file}")

# 데이터 파일 경로 설정 (RunPod 환경에 맞게 조정)
TRAIN_DATA_PATH = "master_dataset_final.csv"  # 업로드된 훈련 데이터
TEST_DATA_PATH = "1차모델_테스트데이터셋.csv"   # 업로드된 테스트 데이터

# 파일 존재 확인
if not os.path.exists(TRAIN_DATA_PATH):
    print(f"❌ 훈련 데이터 파일을 찾을 수 없습니다: {TRAIN_DATA_PATH}")
    print("RunPod 환경에 master_dataset_final.csv 파일을 업로드해주세요.")
else:
    print(f"✅ 훈련 데이터 파일 확인: {TRAIN_DATA_PATH}")

if not os.path.exists(TEST_DATA_PATH):
    print(f"❌ 테스트 데이터 파일을 찾을 수 없습니다: {TEST_DATA_PATH}")
    print("RunPod 환경에 1차모델_테스트데이터셋.csv 파일을 업로드해주세요.")
else:
    print(f"✅ 테스트 데이터 파일 확인: {TEST_DATA_PATH}")

In [ ]:
# 훈련 데이터 로딩
train_df = pd.read_csv(TRAIN_DATA_PATH)
print(f"훈련 데이터 수: {len(train_df)}")
print(f"피싱: {train_df['is_phishing'].sum()}, 일반: {len(train_df) - train_df['is_phishing'].sum()}")

# 테스트 데이터 로딩
test_df = pd.read_csv(TEST_DATA_PATH)
print(f"\n테스트 데이터 수: {len(test_df)}")
print(f"피싱: {test_df['is_phishing'].sum()}, 일반: {len(test_df) - test_df['is_phishing'].sum()}")

# 데이터 미리보기
print("\n=== 훈련 데이터 구조 ===")
print(train_df.columns.tolist())
print(train_df.head(3))

print("\n=== 테스트 데이터 구조 ===")
print(test_df.columns.tolist())
print(test_df.head(3))

## 3. 데이터 전처리 및 대화 시퀀스 생성

In [ ]:
# 대화 시퀀스 생성 함수
def create_dialogue_sequences_from_data(df, text_column='text', label_column='is_phishing', id_column=None):
    """데이터프레임에서 대화 시퀀스 생성"""
    dialogues = []
    
    for idx, row in df.iterrows():
        full_text = str(row[text_column])
        sentences = [s.strip() for s in full_text.split('.') if s.strip()]
        label = int(row[label_column])
        
        if len(sentences) == 0:
            sentences = [full_text]
        
        file_id = row.get(id_column, f"data_{idx}") if id_column and id_column in df.columns else f"data_{idx}"
        
        dialogues.append({
            'file_id': str(file_id),
            'texts': sentences,
            'label': label
        })
    
    return dialogues

# 훈련 및 테스트 대화 시퀀스 생성
train_dialogues = create_dialogue_sequences_from_data(
    train_df, 
    text_column='text', 
    label_column='is_phishing',
    id_column='file_id' if 'file_id' in train_df.columns else None
)

test_dialogues = create_dialogue_sequences_from_data(
    test_df,
    text_column='text',
    label_column='is_phishing',
    id_column='file_name' if 'file_name' in test_df.columns else None
)

print(f"생성된 훈련 대화 시퀀스 수: {len(train_dialogues)}")
print(f"평균 문장 수: {np.mean([len(d['texts']) for d in train_dialogues]):.2f}")
print(f"생성된 테스트 데이터 수: {len(test_dialogues)}")

# 클래스 분포 확인
train_labels = [d['label'] for d in train_dialogues]
test_labels = [d['label'] for d in test_dialogues]

print(f"\n훈련 데이터 클래스 분포: {np.bincount(train_labels)}")
print(f"테스트 데이터 클래스 분포: {np.bincount(test_labels)}")

## 4. KoBERT 모델 및 데이터셋 클래스 정의

In [ ]:
# KoBERT 모델 로딩
MODEL_NAME = "skt/kobert-base-v1"
print(f"KoBERT 모델 로딩 중: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
kobert_model = AutoModel.from_pretrained(MODEL_NAME)

print(f"토크나이저 어휘 크기: {tokenizer.vocab_size}")
print(f"KoBERT 숨겨진 크기: {kobert_model.config.hidden_size}")

In [ ]:
class TextOnlyDialogueDataset(Dataset):
    def __init__(self, dialogues, tokenizer, max_length=128, max_turns=50):
        self.dialogues = dialogues
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.max_turns = max_turns
    
    def __len__(self):
        return len(self.dialogues)
    
    def __getitem__(self, idx):
        dialogue = self.dialogues[idx]
        texts = dialogue['texts'][:self.max_turns]
        label = dialogue['label']
        
        input_ids_list = []
        attention_mask_list = []
        
        for text in texts:
            text = str(text).strip()
            if len(text) == 0:
                text = "[EMPTY]"
            
            encoded = self.tokenizer(
                text,
                max_length=self.max_length,
                padding='max_length',
                truncation=True,
                return_tensors='pt'
            )
            
            input_ids_list.append(encoded['input_ids'].squeeze(0))
            attention_mask_list.append(encoded['attention_mask'].squeeze(0))
        
        return {
            'input_ids': torch.stack(input_ids_list),
            'attention_mask': torch.stack(attention_mask_list),
            'label': torch.tensor(label, dtype=torch.long),
            'num_turns': len(texts)
        }

def collate_fn_text_only(batch):
    max_turns = max([item['num_turns'] for item in batch])
    
    batch_input_ids = []
    batch_attention_mask = []
    batch_labels = []
    batch_lengths = []
    
    for item in batch:
        num_turns = item['num_turns']
        
        if num_turns < max_turns:
            pad_size = max_turns - num_turns
            pad_input_ids = torch.zeros(pad_size, item['input_ids'].size(1), dtype=torch.long)
            pad_attention_mask = torch.zeros(pad_size, item['attention_mask'].size(1), dtype=torch.long)
            
            input_ids = torch.cat([item['input_ids'], pad_input_ids], dim=0)
            attention_mask = torch.cat([item['attention_mask'], pad_attention_mask], dim=0)
        else:
            input_ids = item['input_ids']
            attention_mask = item['attention_mask']
        
        batch_input_ids.append(input_ids)
        batch_attention_mask.append(attention_mask)
        batch_labels.append(item['label'])
        batch_lengths.append(num_turns)
    
    return {
        'input_ids': torch.stack(batch_input_ids),
        'attention_mask': torch.stack(batch_attention_mask),
        'labels': torch.stack(batch_labels),
        'lengths': torch.tensor(batch_lengths, dtype=torch.long)
    }

print("텍스트 전용 데이터셋 클래스 정의 완료")

## 5. GPU 최적화된 텍스트 전용 모델 정의

In [ ]:
class TextOnlyPhishingDetector(nn.Module):
    def __init__(self, kobert_model, hidden_size=256, num_classes=2, dropout=0.3):
        super(TextOnlyPhishingDetector, self).__init__()
        
        self.kobert = kobert_model
        self.kobert_hidden_size = kobert_model.config.hidden_size
        
        # GPU 메모리 효율성을 위한 그라디언트 체크포인팅
        self.kobert.gradient_checkpointing_enable()
        
        self.sentence_projection = nn.Linear(
            self.kobert_hidden_size,
            hidden_size
        )
        
        self.dialogue_lstm = nn.LSTM(
            hidden_size, 
            hidden_size // 2, 
            batch_first=True, 
            bidirectional=True,
            dropout=dropout if hidden_size > 1 else 0
        )
        
        self.attention = nn.MultiheadAttention(
            embed_dim=hidden_size,
            num_heads=8,
            dropout=dropout,
            batch_first=True
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size // 2, num_classes)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        if isinstance(self.sentence_projection, nn.Linear):
            nn.init.xavier_uniform_(self.sentence_projection.weight)
            nn.init.zeros_(self.sentence_projection.bias)
        
        for layer in self.classifier:
            if isinstance(layer, nn.Linear):
                nn.init.xavier_uniform_(layer.weight)
                nn.init.zeros_(layer.bias)
    
    def forward(self, input_ids, attention_mask, lengths):
        batch_size, max_turns, seq_len = input_ids.size()
        
        input_ids_flat = input_ids.view(-1, seq_len)
        attention_mask_flat = attention_mask.view(-1, seq_len)
        
        # KoBERT는 그라디언트를 계산하지 않음 (메모리 절약)
        with torch.no_grad():
            kobert_outputs = self.kobert(
                input_ids=input_ids_flat,
                attention_mask=attention_mask_flat
            )
        
        sentence_embeddings = kobert_outputs.last_hidden_state[:, 0, :]
        sentence_embeddings = sentence_embeddings.view(batch_size, max_turns, -1)
        
        sentence_features = self.sentence_projection(sentence_embeddings)
        
        max_len = max_turns
        padding_mask = torch.arange(max_len, device=lengths.device).expand(
            batch_size, max_len
        ) >= lengths.unsqueeze(1)
        
        lstm_out, _ = self.dialogue_lstm(sentence_features)
        
        attended_out, attention_weights = self.attention(
            lstm_out, lstm_out, lstm_out,
            key_padding_mask=padding_mask
        )
        
        mask = ~padding_mask.unsqueeze(-1)
        masked_attended = attended_out * mask
        dialogue_repr = masked_attended.sum(dim=1) / lengths.unsqueeze(-1).float()
        
        logits = self.classifier(dialogue_repr)
        
        return {
            'logits': logits,
            'attention_weights': attention_weights,
            'dialogue_repr': dialogue_repr
        }

model = TextOnlyPhishingDetector(kobert_model)
model.to(device)

print(f"모델 파라미터 수: {sum(p.numel() for p in model.parameters()):,}")
print(f"훈련 가능한 파라미터 수: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print("GPU 최적화된 텍스트 전용 모델 초기화 완료")

## 6. GPU 최적화된 훈련 설정

In [ ]:
# 클래스 가중치 계산
def calculate_class_weights(labels):
    from sklearn.utils.class_weight import compute_class_weight
    import numpy as np
    
    unique_classes = np.unique(labels)
    class_weights = compute_class_weight(
        'balanced', 
        classes=unique_classes, 
        y=labels
    )
    
    class_weight_dict = dict(zip(unique_classes, class_weights))
    print(f"클래스 분포: {np.bincount(labels)}")
    print(f"클래스 가중치: {class_weight_dict}")
    
    weight_tensor = torch.FloatTensor([class_weights[0], class_weights[1]])
    return weight_tensor

# 전체 훈련 데이터의 클래스 가중치 계산
all_labels = np.array(train_labels)
class_weights = calculate_class_weights(all_labels)

print(f"적용될 클래스 가중치: Normal={class_weights[0]:.3f}, Phishing={class_weights[1]:.3f}")

# 훈련 하이퍼파라미터
K_FOLDS = 5
BATCH_SIZE = 4 if torch.cuda.is_available() else 2  # GPU 메모리에 따라 조정
NUM_EPOCHS = 8
LEARNING_RATE = 2e-5

print(f"\n=== 훈련 설정 ===")
print(f"K-Fold 수: {K_FOLDS}")
print(f"배치 크기: {BATCH_SIZE}")
print(f"에포크 수: {NUM_EPOCHS}")
print(f"학습률: {LEARNING_RATE}")
print(f"Mixed Precision: {use_amp}")

## 7. GPU 최적화된 훈련 및 검증 함수

In [ ]:
def train_epoch_gpu(model, train_loader, criterion, optimizer, device, use_amp=False, scaler=None):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    
    progress_bar = tqdm(train_loader, desc='Training', leave=False)
    
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        lengths = batch['lengths'].to(device, non_blocking=True)
        
        optimizer.zero_grad()
        
        if use_amp and scaler:
            with autocast():
                outputs = model(input_ids, attention_mask, lengths)
                logits = outputs['logits']
                loss = criterion(logits, labels)
            
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(input_ids, attention_mask, lengths)
            logits = outputs['logits']
            loss = criterion(logits, labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
        
        total_loss += loss.item()
        _, predicted = torch.max(logits.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Acc': f'{100. * correct / total:.2f}%'
        })
        
        # GPU 메모리 정리
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return total_loss / len(train_loader), 100. * correct / total

def validate_epoch_gpu(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []
    
    progress_bar = tqdm(val_loader, desc='Validation', leave=False)
    
    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            lengths = batch['lengths'].to(device, non_blocking=True)
            
            outputs = model(input_ids, attention_mask, lengths)
            logits = outputs['logits']
            
            loss = criterion(logits, labels)
            total_loss += loss.item()
            
            _, predicted = torch.max(logits.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            progress_bar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100. * correct / total:.2f}%'
            })
    
    return total_loss / len(val_loader), 100. * correct / total, all_predictions, all_labels

print("GPU 최적화된 훈련 및 검증 함수 정의 완료")

## 8. K-Fold 교차검증 실행

In [ ]:
from sklearn.model_selection import StratifiedKFold

# K-Fold 설정
file_ids = [d['file_id'] for d in train_dialogues]
labels = [d['label'] for d in train_dialogues]

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

print(f"K-Fold 교차검증 시작... (총 {K_FOLDS} 폴드)")
print(f"훈련 대화 수: {len(train_dialogues)}")

fold_results = []
all_val_accs = []

for fold, (train_indices, val_indices) in enumerate(skf.split(file_ids, labels)):
    print(f"\n{'='*60}")
    print(f"Fold {fold+1}/{K_FOLDS} 시작")
    print(f"{'='*60}")
    
    # GPU 메모리 정리
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    fold_train_dialogues = [train_dialogues[i] for i in train_indices]
    fold_val_dialogues = [train_dialogues[i] for i in val_indices]
    
    print(f"Fold {fold+1} - 훈련: {len(fold_train_dialogues)}, 검증: {len(fold_val_dialogues)}")
    
    # 현재 fold의 클래스 분포 확인 및 가중치 계산
    fold_train_labels = [fold_train_dialogues[i]['label'] for i in range(len(fold_train_dialogues))]
    fold_class_weights = calculate_class_weights(np.array(fold_train_labels))
    
    fold_train_dataset = TextOnlyDialogueDataset(fold_train_dialogues, tokenizer)
    fold_val_dataset = TextOnlyDialogueDataset(fold_val_dialogues, tokenizer)
    
    fold_train_loader = DataLoader(
        fold_train_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=True, 
        collate_fn=collate_fn_text_only,
        pin_memory=True if torch.cuda.is_available() else False,
        num_workers=2 if torch.cuda.is_available() else 0
    )
    fold_val_loader = DataLoader(
        fold_val_dataset, 
        batch_size=BATCH_SIZE, 
        shuffle=False, 
        collate_fn=collate_fn_text_only,
        pin_memory=True if torch.cuda.is_available() else False,
        num_workers=2 if torch.cuda.is_available() else 0
    )
    
    # 새로운 모델 인스턴스 생성
    fold_model = TextOnlyPhishingDetector(kobert_model).to(device)
    
    # 클래스 가중치를 적용한 손실 함수
    criterion = nn.CrossEntropyLoss(weight=fold_class_weights.to(device))
    
    optimizer = optim.AdamW(fold_model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3
    )
    
    fold_train_accs = []
    fold_val_accs = []
    best_val_acc = 0
    
    for epoch in range(NUM_EPOCHS):
        print(f"\nFold {fold+1}, Epoch {epoch+1}/{NUM_EPOCHS}")
        
        train_loss, train_acc = train_epoch_gpu(
            fold_model, fold_train_loader, criterion, optimizer, device, 
            use_amp=use_amp, scaler=scaler if use_amp else None
        )
        val_loss, val_acc, val_preds, val_labels = validate_epoch_gpu(
            fold_model, fold_val_loader, criterion, device
        )
        
        scheduler.step(val_loss)
        
        fold_train_accs.append(train_acc)
        fold_val_accs.append(val_acc)
        
        if val_acc > best_val_acc:
            best_val_acc = val_acc
        
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")
        print(f"Current LR: {optimizer.param_groups[0]['lr']:.2e}")
        
        # GPU 메모리 상태 출력
        if torch.cuda.is_available():
            memory_used = torch.cuda.memory_allocated() / 1024**3
            memory_cached = torch.cuda.memory_reserved() / 1024**3
            print(f"GPU Memory: {memory_used:.1f}GB / {memory_cached:.1f}GB")
    
    print(f"\nFold {fold+1} 최고 검증 정확도: {best_val_acc:.2f}%")
    
    fold_results.append({
        'fold': fold+1,
        'best_val_acc': best_val_acc,
        'final_val_acc': fold_val_accs[-1],
        'train_accs': fold_train_accs,
        'val_accs': fold_val_accs,
        'val_predictions': val_preds,
        'val_labels': val_labels
    })
    all_val_accs.append(best_val_acc)
    
    # Fold 완료 후 메모리 정리
    del fold_model, fold_train_loader, fold_val_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# K-Fold 결과 요약
print(f"\n{'='*60}")
print("K-Fold 교차검증 결과 요약")
print(f"{'='*60}")

for i, result in enumerate(fold_results):
    print(f"Fold {i+1}: {result['best_val_acc']:.2f}%")

mean_acc = np.mean(all_val_accs)
std_acc = np.std(all_val_accs)

print(f"\n평균 검증 정확도: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"최고 검증 정확도: {max(all_val_accs):.2f}%")
print(f"최저 검증 정확도: {min(all_val_accs):.2f}%")

## 9. 전체 데이터로 최종 모델 훈련

In [ ]:
print(f"\n{'='*60}")
print("전체 데이터로 최종 모델 훈련")
print(f"{'='*60}")

# GPU 메모리 정리
if torch.cuda.is_available():
    torch.cuda.empty_cache()

final_train_dataset = TextOnlyDialogueDataset(train_dialogues, tokenizer)
final_train_loader = DataLoader(
    final_train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn_text_only,
    pin_memory=True if torch.cuda.is_available() else False,
    num_workers=2 if torch.cuda.is_available() else 0
)

final_model = TextOnlyPhishingDetector(kobert_model).to(device)

# 전체 데이터의 클래스 가중치 적용
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = optim.AdamW(final_model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

NUM_FINAL_EPOCHS = 8
for epoch in range(NUM_FINAL_EPOCHS):
    print(f"\nFinal Training Epoch {epoch+1}/{NUM_FINAL_EPOCHS}")
    train_loss, train_acc = train_epoch_gpu(
        final_model, final_train_loader, criterion, optimizer, device,
        use_amp=use_amp, scaler=scaler if use_amp else None
    )
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%")
    
    if torch.cuda.is_available():
        memory_used = torch.cuda.memory_allocated() / 1024**3
        print(f"GPU Memory: {memory_used:.1f}GB")

print("\n최종 모델 훈련 완료!")

## 10. 테스트 데이터 평가

In [ ]:
# 테스트 데이터 평가
test_dataset = TextOnlyDialogueDataset(test_dialogues, tokenizer)
test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn_text_only,
    pin_memory=True if torch.cuda.is_available() else False,
    num_workers=2 if torch.cuda.is_available() else 0
)

print(f"\n테스트 데이터 평가 시작... (테스트 샘플 수: {len(test_dialogues)})")

final_model.eval()
test_correct = 0
test_total = 0
test_predictions = []
test_labels = []
test_probabilities = []

progress_bar = tqdm(test_loader, desc='Testing')

with torch.no_grad():
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device, non_blocking=True)
        attention_mask = batch['attention_mask'].to(device, non_blocking=True)
        labels = batch['labels'].to(device, non_blocking=True)
        lengths = batch['lengths'].to(device, non_blocking=True)
        
        outputs = final_model(input_ids, attention_mask, lengths)
        logits = outputs['logits']
        
        probs = torch.softmax(logits, dim=1)
        
        _, predicted = torch.max(logits.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
        test_predictions.extend(predicted.cpu().numpy())
        test_labels.extend(labels.cpu().numpy())
        test_probabilities.extend(probs.cpu().numpy())
        
        progress_bar.set_postfix({
            'Acc': f'{100. * test_correct / test_total:.2f}%'
        })

test_accuracy = 100. * test_correct / test_total

print(f"\n{'='*60}")
print("최종 테스트 결과")
print(f"{'='*60}")
print(f"K-Fold 평균 검증 정확도: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"테스트 정확도: {test_accuracy:.2f}%")

print("\n테스트 데이터 분류 리포트:")
print(classification_report(test_labels, test_predictions, target_names=['Normal', 'Phishing']))

# 클래스별 성능 분석
from sklearn.metrics import precision_recall_fscore_support
precision, recall, f1, support = precision_recall_fscore_support(test_labels, test_predictions)

print(f"\n=== 클래스별 상세 성능 ===")
print(f"Normal   - Precision: {precision[0]:.3f}, Recall: {recall[0]:.3f}, F1: {f1[0]:.3f}")
print(f"Phishing - Precision: {precision[1]:.3f}, Recall: {recall[1]:.3f}, F1: {f1[1]:.3f}")

# ROC AUC 계산
test_probs = np.array(test_probabilities)[:, 1]
fpr, tpr, _ = roc_curve(test_labels, test_probs)
roc_auc = auc(fpr, tpr)
print(f"\n테스트 AUC: {roc_auc:.3f}")

# 데이터 불균형 대응 효과 분석
print(f"\n=== 데이터 불균형 대응 효과 ===")
print(f"적용된 클래스 가중치 - Normal: {class_weights[0]:.3f}, Phishing: {class_weights[1]:.3f}")
print(f"테스트 데이터 클래스 분포: {np.bincount(test_labels)}")
print(f"Balanced Accuracy: {np.mean([recall[0], recall[1]]):.3f}")

## 11. 결과 시각화

In [ ]:
# 결과 시각화
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Fold별 성능 비교
folds = [f"Fold {i+1}" for i in range(K_FOLDS)]
ax1.bar(folds, all_val_accs, alpha=0.7, color="skyblue")
ax1.axhline(y=mean_acc, color="red", linestyle="--", label=f"평균: {mean_acc:.2f}%")
ax1.set_title("Fold별 검증 정확도 (GPU 최적화)")
ax1.set_xlabel("Fold")
ax1.set_ylabel("Accuracy (%)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# 테스트 confusion matrix
cm = confusion_matrix(test_labels, test_predictions)
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    ax=ax2,
    xticklabels=["Normal", "Phishing"],
    yticklabels=["Normal", "Phishing"],
)
ax2.set_title(f"Test Confusion Matrix (Accuracy: {test_accuracy:.2f}%)")
ax2.set_xlabel("Predicted")
ax2.set_ylabel("Actual")

# ROC Curve
ax3.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
ax3.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--")
ax3.set_xlim([0.0, 1.0])
ax3.set_ylim([0.0, 1.05])
ax3.set_xlabel("False Positive Rate")
ax3.set_ylabel("True Positive Rate")
ax3.set_title("ROC Curve")
ax3.legend(loc="lower right")
ax3.grid(True, alpha=0.3)

# 클래스별 성능 비교
metrics = ["Precision", "Recall", "F1-Score"]
normal_scores = [precision[0], recall[0], f1[0]]
phishing_scores = [precision[1], recall[1], f1[1]]

x = np.arange(len(metrics))
width = 0.35

ax4.bar(x - width / 2, normal_scores, width, label="Normal", alpha=0.7, color="blue")
ax4.bar(x + width / 2, phishing_scores, width, label="Phishing", alpha=0.7, color="red")
ax4.set_xlabel("Metrics")
ax4.set_ylabel("Score")
ax4.set_title("클래스별 성능 지표 (GPU 최적화)")
ax4.set_xticks(x)
ax4.set_xticklabels(metrics)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("runpod_2nd_model_results.png", dpi=300, bbox_inches="tight")
plt.show()

print("결과 시각화 완료! 'runpod_2nd_model_results.png' 파일로 저장되었습니다.")

## 12. 최종 모델 저장

In [ ]:
# 최종 모델 저장
final_model_path = "kobert_2nd_model_runpod.pth"
torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "model_config": {"hidden_size": 256, "num_classes": 2, "dropout": 0.3},
        "tokenizer_name": MODEL_NAME,
        "class_weights": class_weights.tolist(),
        "kfold_results": {
            "mean_val_acc": mean_acc,
            "std_val_acc": std_acc,
            "fold_accs": all_val_accs,
            "fold_details": fold_results,
        },
        "test_accuracy": test_accuracy,
        "test_auc": roc_auc,
        "balanced_accuracy": np.mean([recall[0], recall[1]]),
        "training_info": {
            "device": str(device),
            "batch_size": BATCH_SIZE,
            "epochs": NUM_EPOCHS,
            "learning_rate": LEARNING_RATE,
            "mixed_precision": use_amp,
            "gpu_optimized": True
        }
    },
    final_model_path,
)

print(f"\n최종 모델이 저장되었습니다: {final_model_path}")
print(f"파일 크기: {os.path.getsize(final_model_path) / (1024*1024):.1f} MB")

## 13. 훈련 완료 요약

In [ ]:
# GPU 최적화 효과 분석
if torch.cuda.is_available():
    gpu_info = {
        'name': torch.cuda.get_device_name(0),
        'memory_total': torch.cuda.get_device_properties(0).total_memory / 1024**3,
        'memory_used': torch.cuda.memory_allocated() / 1024**3,
        'memory_cached': torch.cuda.memory_reserved() / 1024**3
    }
else:
    gpu_info = {'name': 'CPU Only'}

# 최종 결과 요약
print("\n" + "=" * 70)
print("RunPod GPU 환경 2차 모델 훈련 완료 요약")
print("=" * 70)
print(f"🖥️  실행 환경: {gpu_info['name'] if 'name' in gpu_info else 'Unknown'}")
print(f"🔧  모델 아키텍처: KoBERT + LSTM + Attention (화자 정보 제거)")
print(f"📊  교차검증: {K_FOLDS}-Fold Stratified")
print(f"🎯  훈련 데이터: {len(train_dialogues)} 대화")
print(f"🧪  테스트 데이터: {len(test_dialogues)} 샘플")
print(f"")
print(f"⚖️  === 불균형 데이터 대응 ===")
print(f"     클래스 가중치: Normal={class_weights[0]:.3f}, Phishing={class_weights[1]:.3f}")
print(f"")
print(f"🚀  === GPU 최적화 설정 ===")
print(f"     배치 크기: {BATCH_SIZE}")
print(f"     Mixed Precision: {use_amp}")
print(f"     그라디언트 체크포인팅: 활성화")
if torch.cuda.is_available():
    print(f"     GPU 메모리 사용량: {gpu_info['memory_used']:.1f}GB / {gpu_info['memory_total']:.1f}GB")
print(f"")
print(f"📈  === 성능 결과 ===")
print(f"     K-Fold 평균 검증 정확도: {mean_acc:.2f}% ± {std_acc:.2f}%")
print(f"     테스트 정확도: {test_accuracy:.2f}%")
print(f"     테스트 AUC: {roc_auc:.3f}")
print(f"     Balanced Accuracy: {np.mean([recall[0], recall[1]]):.3f}")
print(f"")
print(f"🎯  === 클래스별 성능 ===")
print(f"     Normal   - Precision: {precision[0]:.3f}, Recall: {recall[0]:.3f}, F1: {f1[0]:.3f}")
print(f"     Phishing - Precision: {precision[1]:.3f}, Recall: {recall[1]:.3f}, F1: {f1[1]:.3f}")
print(f"")
print(f"✅  성능 안정성: 표준편차 {std_acc:.2f}%로 {'안정적' if std_acc < 2.0 else '다소 불안정'}")
print(f"🎉  GPU 최적화로 효율적인 대규모 모델 훈련 완료!")
print(f"💾  모델 저장 위치: {final_model_path}")
print("=" * 70)

# GPU 메모리 최종 정리
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("\n🧹 GPU 메모리 정리 완료")

print("\n🚀 RunPod 환경에서의 GPU 최적화된 훈련이 성공적으로 완료되었습니다!")